> 📓 **Lesson 1.10 — Part 2 of 4: Matplotlib Fundamentals**
>
> This notebook was split out of the original single `data_visualization_lesson.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.10.
>
> Other notebooks in this set: `Part_1_three_pillars.ipynb`, `Part_3_seaborn.ipynb`, `Part_4_chart_choice.ipynb`

# Lesson 1.10: Data Visualisation & Storytelling

Lesson 1.8 asked *can I trust this data?* Lesson 1.9 asked *what is the pattern?* You answered both.
You have a table that shows exactly what is happening to the café chain.

This lesson is about the last twenty seconds — the part where the owner either acts on it or doesn't.

**Structure — the four learning outcomes, in order:**
* **Part 1: The Three Pillars** — *apply* perception, design and storytelling to fix a bad chart.
* **Part 2: Matplotlib Fundamentals** — *create* charts with the Figure → Axes → plot hierarchy.
* **Part 3: Seaborn for Statistical Graphics** — *use* the right statistical view for the data type.
* **Part 4: Chart Choice & the One Slide** — *select* the chart your message needs, and build it.

**How to work through this:** read the `# 👉` comment above each line before you run the cell. Charts are the one
topic where you learn most by breaking things on purpose, so several cells here are deliberately bad.


> **🧭 Today's flow — 180 minutes.** One message, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | The Three Pillars | **Apply** perception, design, storytelling | 40 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Matplotlib Fundamentals | **Create** charts: Figure → Axes → plot | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Seaborn | **Use** statistical graphics for the data type | 45 min |
> | **Part 4** | Chart Choice & the One Slide | **Select** the chart the message needs | 25 min |
>
> **The spine:** the same business problem as 1.9 — *The Daily Grind* café chain — and the same
> files. You already know the answer. Today you make it land.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives (colour theory, accessibility,
> Gestalt, the research behind pre-attentive processing) live in `reference.md`.


### The one rule this lesson is built on

> **The chart is chosen by the message, not by taste.**

Write the sentence you want the audience to leave with, in words, first. The chart type follows from
that sentence almost mechanically. Every mistake in Part 1 comes from doing it the other way round —
picking a chart, then hunting for something to say.


### Setup


In [ ]:
# 👉 pandas for the data, matplotlib for drawing, seaborn for statistical charts.
#    `plt` and `sns` are the conventional nicknames -- you will see them in every tutorial.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 👉 Housekeeping only: keep version-deprecation notices out of our output.
import warnings
warnings.filterwarnings("ignore")

# 👉 Draw charts underneath the cell that made them, inside the notebook.
%matplotlib inline


In [ ]:
# 👉 The same café data as Lesson 1.9, plus the decision table you produced at the end of it.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])
tickets = pd.read_csv("../data/tickets_week.csv", parse_dates=["txn_datetime"])
decision = pd.read_csv("../data/lesson19_decision.csv", index_col="outlet_id")

decision.round(1)


In [ ]:
# 👉 Two summaries we will chart all session. Both are straight out of Lesson 1.9.
#    (1) monthly revenue per outlet -- one column per outlet, dates on the index.
monthly = sales.pivot_table(
    index="date", columns="outlet_id", values="revenue_sgd", aggfunc="sum"
).resample("M").sum()

# 👉 (2) revenue by outlet and daypart.
by_daypart = sales.pivot_table(
    index="outlet_id", columns="daypart", values="revenue_sgd", aggfunc="sum"
)[["Morning", "Midday", "Evening"]]

# 👉 Friendly names, for chart labels. Codes belong in databases, not on slides.
names = outlets.set_index("outlet_id")["outlet_name"].to_dict()

monthly.tail(5).round(0)

# by_daypart.round(0)


### 🎬 Why this matters — the same finding, three ways

**The situation.** You have five minutes at the end of a Monday meeting. The owner is deciding
whether to renew the Marina Bay lease. You already know the answer: chain revenue is flat because
Marina Bay is falling and Holland Village is rising, and Marina Bay's fall is a step dated to the
first week of November 2024.

Below is that finding presented three ways. Same data. Same truth. Run all three, then decide which
one makes somebody *do* something.


In [ ]:
# 👉 Version 1: the numbers. Complete, precise, and almost impossible to read at speed.
monthly.round(0).tail(12)


In [ ]:
# 👉 Version 2: a chart, made without thinking. Every default left as it came.
monthly.plot()
plt.show()


In [ ]:
# 👉 Version 3: the same numbers, drawn to make one point. Read the code later --
#    for now just compare what your eye does with this versus the two above.
# color codes: https://htmlcolorcodes.com/color-names/

fig, ax = plt.subplots(figsize=(9, 4.5))

for col in monthly.columns:
    if col == "OUT-03":
        ax.plot(monthly.index, monthly[col], color="#c0392b", linewidth=2.8, label="Marina Bay", zorder=3) # zorder controls which lines are drawn on top of which others, 3 is the highest here
    elif col == "OUT-04":
        ax.plot(monthly.index, monthly[col], color="#27ae60", linewidth=2.2, label="Holland Village", zorder=2)
    else:
        ax.plot(monthly.index, monthly[col], color="#c8c8c8", linewidth=1.4, zorder=1)

ax.axvline(pd.Timestamp("2024-11-04"), color="#7f8c8d", linestyle="--", linewidth=1)
ax.annotate("competitor opens\nnext door, 4 Nov",
            xy=(pd.Timestamp("2024-11-04"), 47000), xytext=(pd.Timestamp("2024-11-20"), 55000),
            fontsize=9, color="#7f8c8d",
            arrowprops=dict(arrowstyle="->", color="#7f8c8d", linewidth=0.8))

ax.set_title("Marina Bay is falling; Holland Village is covering it up", fontsize=13, weight="bold")
ax.set_ylabel("monthly revenue (S$)")
ax.set_ylim(0, 60000)
ax.legend(frameon=False, loc="lower left") # remove the box around the legend
ax.spines[["top", "right"]].set_visible(False) # remove the top and right borders
plt.show()


**What changed between versions 2 and 3?** Nothing about the data. Four things about the drawing:

1. **One sentence as the title.** Not "Monthly Revenue" — the actual finding, in words.
2. **Colour used as an argument.** Two lines carry the point; the other three went grey. Your eye
   found the red line before you consciously read anything. That reflex is *pre-attentive processing*,
   and it fires in 200 to 500 milliseconds. Charts either use it or waste it.
3. **The cause annotated on the chart.** The reader does not have to be told separately.
4. **Clutter removed.** Two spines gone, no box round the legend, no chart junk competing for attention.

That is the whole lesson: **perception** (how eyes work), **design** (honest, uncluttered choices),
**storytelling** (one message, stated). Everything else is syntax.


## Part 2: Matplotlib Fundamentals

**Learning outcome 2:** *Create charts using Matplotlib's Figure-Axes-Plot hierarchy with proper
customisation.*

**Goal:** stop fighting the defaults. Once the hierarchy clicks, every tutorial you read afterwards
makes sense.

⏱️ ~45 min including Group Exercise 2


### 2.1: Figure, Axes, plot — the whole mental model

Three levels, and almost every error comes from confusing them:

| Level | What it is | The analogy |
|---|---|---|
| **Figure** | the whole image, the thing you save | the sheet of paper |
| **Axes** | one set of x and y axes — one panel | a chart drawn on the paper |
| **plot / artists** | the marks: lines, bars, text | the ink |

A Figure can hold many Axes. `plt.subplots()` creates both at once, and returns them in that order:
`fig, ax = plt.subplots()`. **Learn that line.** Everything else hangs off `ax`.


In [ ]:
# 👉 The canonical opening line: one Figure, one Axes. `figsize` is in inches (width, height). 
fig, ax = plt.subplots(figsize=(7, 3.5))

# 👉 `ax.plot(x, y)` draws a line on that Axes. Here: chain revenue per month.
chain = monthly.sum(axis=1)
ax.plot(chain.index, chain.values)

# 👉 Everything descriptive is a method on `ax`. Same pattern for all of them.
ax.set_title("Chain revenue is flat -- and that is the problem")
ax.set_ylabel("monthly revenue (S$)")

plt.show()


In [ ]:
# 👉 `ax` also has the styling controls. Four habits worth using every time:
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(chain.index, chain.values, color="#2c3e50", linewidth=2)

ax.set_ylim(0, chain.max() * 1.15)                      # 1. include zero for magnitude
ax.spines[["top", "right"]].set_visible(False)          # 2. remove the box
ax.grid(axis="y", alpha=0.3)                            # 3. faint horizontal guides only
ax.set_title("Chain revenue, Jan 2024 - Jun 2025", loc="left", weight="bold")   # 4. title left

plt.show()


> **`plt.something()` versus `ax.something()`.** The `plt.` functions act on "the current chart",
> which is fine for one panel and a menace with several. `ax.` is explicit about *which* panel.
> Use the `fig, ax = plt.subplots()` style from the start and you will never debug this.


### 2.2: Subplots — several Axes on one Figure

Small multiples are the most underrated chart type in business reporting: the same chart repeated per
group, on identical axes, so differences in *shape* are obvious.


In [ ]:
# 👉 A 2x2 grid. `axes` comes back as a NumPy array of Axes, so `.flatten()` lets us loop over it.
#    `sharey=True` forces one scale on all four -- essential, or the panels lie by comparison.
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey=True)

for ax, outlet in zip(axes.flatten(), ["OUT-01", "OUT-02", "OUT-03", "OUT-04"]):
    ax.plot(monthly.index, monthly[outlet], color="#2c3e50", linewidth=1.8)
    ax.fill_between(monthly.index, monthly[outlet], color="#2c3e50", alpha=0.12)
    ax.set_title(names[outlet], fontsize=10)
    ax.set_ylim(0, 60000)
    ax.spines[["top", "right"]].set_visible(False)

# 👉 One title for the whole Figure -- note it is `fig`, not `ax`.
fig.suptitle("Same axes, four outlets: only one is falling", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


> **`sharey=True` is doing the ethical work here.** Without it, matplotlib scales each panel to its
> own data and Holland Village's \$45k month looks the same height as Raffles Place's \$55k month.
> Small multiples without a shared scale are one of the easiest ways to mislead by accident.


### 2.3: Ticks, limits and labels

Axis text is not decoration — it is most of what makes a chart readable at a glance.


In [ ]:
# 👉 Two fixes that pay for themselves on every chart you make:
#    1. thousands separators on the money axis, so nobody counts digits
#    2. horizontal date labels, by using fewer of them
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(monthly.index, monthly["OUT-03"], color="#c0392b", linewidth=2.2)

# 👉 A formatter takes the raw tick value and returns the text to print.
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v/1000:,.0f}k"))

ax.set_xticks(monthly.index[::3])                          # every third month only
ax.set_xticklabels([d.strftime("%b %Y") for d in monthly.index[::3]])

ax.set_ylim(0, 50000)
ax.set_title("Marina Bay monthly revenue", loc="left", weight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.show()


### 2.4: Annotation — putting the sentence on the chart

An annotated chart survives being forwarded without you attached to it. This is the single highest-value
matplotlib skill in a business setting.


In [ ]:
# 👉 `axvline` draws a vertical rule; `annotate` places text with an optional arrow.
#    `xy` is what you are pointing AT; `xytext` is where the words sit.
fig, ax = plt.subplots(figsize=(8.5, 4))

ax.plot(monthly.index, monthly["OUT-03"], color="#c0392b", linewidth=2.4)
ax.axvline(pd.Timestamp("2024-11-04"), color="#95a5a6", linestyle="--", linewidth=1) # vertical rule

ax.annotate("competitor opens next door\n-21% and never recovers",
            xy=(pd.Timestamp("2024-11-04"), 38000),
            xytext=(pd.Timestamp("2024-12-10"), 45000),
            fontsize=10, color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", linewidth=1))

# 👉 Shade the period after the event, so "before" and "after" read as two regimes.
ax.axvspan(pd.Timestamp("2024-11-04"), monthly.index.max(), color="#c0392b", alpha=0.1)

ax.set_ylim(0, 50000)
ax.set_title("Marina Bay: a step, not a slide", loc="left", fontsize=13, weight="bold")
ax.set_ylabel("monthly revenue (S$)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()


In [ ]:
# 👉 Saving. `dpi=150` is fine for slides; `bbox_inches="tight"` stops labels being cut off.
#    Always save from the `fig` object, and save BEFORE `plt.show()` in a script.
import os
os.makedirs("../visualisations", exist_ok=True)
fig.savefig("../visualisations/marina_step.png", dpi=150, bbox_inches="tight")

print("saved visualisations/marina_step.png")


### 🛠️ Group Exercise 2 — Build it yourself (8 min)

Build a bar chart of **revenue per seat** for the four permanent outlets: total 2025 H1 revenue divided by `seats`. Zero baseline, thousands-formatted axis, and a title that states the finding rather than naming the metric.

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Revenue per seat, 2025 H1, four permanent outlets.**

In [ ]:
# 👉 Revenue per seat, 2025 H1, four permanent outlets.
#    Step 1 -- the number. Filter to H1 2025, total the revenue, divide by seats.
h1_sales = sales[(sales["date"] >= "2025-01-01") & (sales["date"] <= "2025-06-30")]
rev_h1 = h1_sales.groupby("outlet_id")["revenue_sgd"].sum()
seats = outlets.set_index("outlet_id")["seats"]

per_seat = (rev_h1 / seats).loc[["OUT-01", "OUT-02", "OUT-03", "OUT-04"]].sort_values(ascending=False)

# 👉 Step 2 -- the chart. Zero baseline (it is a bar), formatted axis, finding as the title.
fig, ax = plt.subplots(figsize=(7.5, 3.4))

ax.bar([names[i] for i in per_seat.index], per_seat.values, color="#34495e")

ax.set_ylim(0, per_seat.max() * 1.18)                                   # bars start at zero
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v/1000:,.0f}k"))
ax.set_ylabel("2025 H1 revenue per seat (S$)")
ax.set_title("Raffles Place works its seats 1.7x as hard as Tampines Mall",
             loc="left", weight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

per_seat.round(0)

**Careful with the title you just wrote.** Revenue per seat ranks Raffles Place \$11.0k, Marina Bay \$8.5k, Holland Village \$7.0k, Tampines Mall \$6.6k — so *Marina Bay is not the worst on this metric*. If you had assumed the finding before computing it and titled the chart "Marina Bay wastes its seats", the chart would have contradicted you in front of the owner.

This is the honest version of chart selection: the metric decides the story, not the other way round. Seats are a proxy for floor area, and mall outlets buy space cheaply, so per-seat revenue is the wrong measure of Marina Bay's problem. **Rent as a share of revenue** (28% vs ~15%) is the right one — and that is why the owner's slide in Part 4 uses that and not this.

---

# ☕ Break — 10 minutes

**Where we are:** you can build a chart from scratch, control its axes, annotate it and save it.

**Next up:** Part 3 — seaborn, for the charts that involve a *distribution* rather than a single
number per group. This is where you stop hiding behind averages.


📂 **Open** `Part_3_seaborn.ipynb` to continue.